In [ ]:
import plotly.express as px
import pandas as pd
import struct

def summarize_all_componentwise_errors(matrix_type, filenames):

    dfs = []
    for filename in filenames:
        input_file = f"results/{matrix_type}/{filename}"
        with open(input_file, "rb") as f:
            byte_buffer = f.read()

        errors = []
        for i in range(0,len(byte_buffer),8):
            errors.append(struct.unpack('d', byte_buffer[i:i+8]))

        df_temp = pd.DataFrame(errors, columns=['error'])
        df_temp["filename"] = filename
        dfs.append(df_temp)

    df = pd.concat(dfs)
    for filename in filenames:
        print(f"{matrix_type}/{filename} stats:")
        print(f"\t   mean: {df[df['filename'] == filename]['error'].mean()}")
        print(f"\t    std: {df[df['filename'] == filename]['error'].std()}")
        print(f"\tn_zeros: {len(df[(df['filename'] == filename) & (df['error'] == 0.0)])}")

    px.histogram(df, x='error', color='filename', log_y=True, title=f"All Componentwise Errors for {matrix_type} matrices").update_traces(opacity=0.5).update_layout(yaxis_tickformat='.2e', xaxis_tickformat='.2e').show()

def plot_max_componentwise_errors_vs_flops(matrix_type, filenames):
    
    dfs = []
    for filename in filenames:
        input_file = f"results/{matrix_type}/{filename}"
        df_temp = pd.read_csv(input_file)
        df_temp["input_file"] = input_file
        dfs.append(df_temp)
    
    df_temp = pd.concat(dfs, ignore_index=True)

    df_temp["FLOPs"] = 2 * df_temp["m"] * df_temp["n"] * df_temp["k"]
    df_temp["Square?"] = (df_temp["m"] == df_temp["n"]) & (df_temp["n"] == df_temp["k"])
    fig = px.scatter(
        df_temp,
        x="FLOPs",
        y="max_componentwise_error",
        log_x=True,
        log_y=True,
        title=f"Max Componentwise Error vs FLOPs for {matrix_type} matrices",
        hover_data=["m", "n", "k"],
        symbol="Square?",
        symbol_sequence=["square", "arrow-wide"],
        color="input_file",
    )
    fig.update_traces(
        marker=dict(
            size=15,
            opacity=0.7,
        )
    )
    fig.update_layout(
        yaxis_tickformat='.2e',
        xaxis_tickformat='.2e'
    )

    fig.show()


In [ ]:
for matrix_type in ["U_0_1", "U_-1_1", "test1"]:
    summarize_all_componentwise_errors(matrix_type, ["all_componentwise_errors.blis.bin", "all_componentwise_errors.reference.bin", "all_componentwise_errors.strassen.bin"])

In [ ]:
for matrix_type in ["U_0_1", "U_-1_1", "test1"]:
    plot_max_componentwise_errors_vs_flops(matrix_type, ["max_componentwise_errors.blis.txt", "max_componentwise_errors.reference.txt", "max_componentwise_errors.strassen.txt"])